# 01. Token, embedding과 sampling 기초

목표: 문자열이 token ID와 vector를 거쳐 다음-token probability가 되는 최소 흐름을 NumPy로 확인합니다. 실제 tokenizer나 LLM을 재현하는 예제가 아닌 작은 모형입니다.

In [ ]:
import re
import numpy as np

rng = np.random.default_rng(7)
np.set_printoptions(precision=3, suppress=True)

## 1. Toy subword tokenizer

실제 BPE나 SentencePiece 대신 longest-match 규칙으로 subword 절충을 관찰합니다. Vocabulary에 없는 문자는 `<unk>`로 처리합니다.

In [ ]:
pieces = ['<unk>', 'how', 'llm', 'work', 'token', 'ization', 'transform', 'er', 's', 'actually']
token_to_id = {piece: index for index, piece in enumerate(pieces)}
lexical_pieces = sorted(pieces[1:], key=len, reverse=True)

def encode_word(word):
    word = word.lower()
    ids = []
    cursor = 0
    while cursor < len(word):
        match = next((p for p in lexical_pieces if word.startswith(p, cursor)), None)
        if match is None:
            ids.append(token_to_id['<unk>'])
            cursor += 1
        else:
            ids.append(token_to_id[match])
            cursor += len(match)
    return ids

def encode(text):
    words = re.findall(r'[A-Za-z]+', text)
    return [token_id for word in words for token_id in encode_word(word)]

prompt = 'How transformers actually work'
ids = encode(prompt)
decoded_pieces = [pieces[i] for i in ids]
print(prompt, '->', decoded_pieces, '->', ids)
assert decoded_pieces == ['how', 'transform', 'er', 's', 'actually', 'work']

## 2. Embedding lookup

ID는 embedding matrix의 row를 고릅니다. 여기서는 학습되지 않은 random vector이므로 semantic geometry를 기대하면 안 됩니다.

In [ ]:
d_model = 8
embedding_matrix = rng.normal(0, 0.2, size=(len(pieces), d_model))
x = embedding_matrix[ids]
print('embedding matrix:', embedding_matrix.shape)
print('prompt representation:', x.shape)
assert x.shape == (len(ids), d_model)
assert np.allclose(x[0], embedding_matrix[token_to_id['how']])

## 3. Logits, temperature와 top-k sampling

Softmax는 raw logit을 probability로 바꿉니다. Temperature가 낮으면 큰 logit의 우위가 커지고, top-k는 후보 수를 제한합니다.

In [ ]:
def softmax(values):
    shifted = values - np.max(values)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()

def sample_top_k(logits, temperature=1.0, top_k=None, generator=rng):
    if temperature <= 0:
        return int(np.argmax(logits)), softmax(logits)
    scaled = logits / temperature
    if top_k is not None and top_k < len(scaled):
        keep = np.argpartition(scaled, -top_k)[-top_k:]
        masked = np.full_like(scaled, -np.inf)
        masked[keep] = scaled[keep]
        scaled = masked
    probabilities = softmax(scaled)
    return int(generator.choice(len(logits), p=probabilities)), probabilities

logits = np.array([-1.0, 0.3, 2.2, 1.1, -0.5])
for temperature in (0.5, 1.0, 2.0):
    selected, probs = sample_top_k(logits, temperature=temperature, top_k=3)
    print(f'T={temperature}: selected={selected}, probs={probs}')
    assert np.isclose(probs.sum(), 1.0)
    assert np.count_nonzero(probs) == 3

greedy, _ = sample_top_k(logits, temperature=0)
assert greedy == int(np.argmax(logits))

## 정리

실제 LLM에서도 큰 흐름은 같습니다. Tokenizer는 text를 ID로 바꾸고, embedding lookup이 vector를 만들며, Transformer가 마지막 위치의 표현을 logits로 변환합니다. Sampling policy는 model weights를 바꾸지 않고 출력 다양성만 조절합니다.